# **Project 1 of Machine Learning**

In [ ]:
# Import all the libraries used in this project
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# To Ignore warnings because aur Project will be in developing state
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# To create a Project, we need a data, so let's fetch the data
myData=pd.read_csv('Project1OfMLinsurance.csv')
myData

# EDA STEPS

**1. VIEWING THE DATA :**

In [ ]:

# To check the data shape
myData.shape

In [ ]:
# To check the data head
myData.head()

In [ ]:
# To check the data information
myData.info()

**2. Summary Statistics**

In [ ]:
# Describe the data
myData.describe()

**4. Missing Value Analysis**

In [ ]:
# To check if there is any null value my data
myData.isnull().sum()

**5. Visualizations**

In [ ]:
# I will check the coloumns for visualization
myData.columns

In [ ]:
# Extracting all numeric columns out of these columns
numericColumns=['age', 'bmi', 'children', 'charges']

# 1. Creating Visuaulizations ( Figures ) : Histogram
for columns in numericColumns:
    plt.figure(figsize=(6,4))
    sns.histplot(myData[columns], kde=True, bins=20)
    

In [ ]:
# Analysing Categorical Datum:  CHILDREN, REGION, SMOKERS, SEX
sns.countplot(x=myData['children'])

In [ ]:
sns.countplot(x = myData['sex'])

In [ ]:
sns.countplot(x = myData['smoker'])

In [ ]:
# 2. Creating Visuaulizations ( Figures ) : Boxplots
for columns in numericColumns:
    plt.figure(figsize= (6,4))
    sns.boxplot(x = myData[columns])

# and now we will see the outliers in all of these

In [ ]:
# 3. Now i will check the CORRELATION -> so i will create a heat map-> heat map used only for numerical value
plt.figure(figsize=(8,6))
sns.heatmap(myData.corr(numeric_only=True),annot=True)

# Data Cleaning And Data PreProcessing

In [ ]:
# Now I will use one more data set to clean the data and remove the outliers from the data set, so I will create a copy of myData
myCleanedData=myData.copy()
myCleanedData.head()

In [ ]:
# I will check the shape of the data
myCleanedData.shape

In [ ]:
# Now i will remove duplicates
myCleanedData.drop_duplicates(inplace=True)
myCleanedData.shape

In [ ]:
# Now i will check the missing values in the data set
myCleanedData.isnull().sum()

In [ ]:
# Printing all the data types
myCleanedData.dtypes

In [ ]:
# To create a ML model, all the categorical data should be converted into numerical data.

# i will check value counts of categorical data: sex
myCleanedData['sex'].value_counts()

In [ ]:
# Now i will encode the categorical data into numerical data using Label encoding for sex
myCleanedData['sex']=myCleanedData['sex'].map({"male" : 0,"female" : 1})
myCleanedData.head()


In [ ]:
# To check how many values of sex
myCleanedData['sex'].value_counts()

In [ ]:
# To check how many values of smoker
myCleanedData['smoker'].value_counts()

In [ ]:
# Now i will encode the categorical data into numerical data using Label encoding for smoker
myCleanedData['smoker']=myCleanedData['smoker'].map({'yes':1,'no':0})
myCleanedData.head()

In [ ]:
# Now i will rename sex and smoker
myCleanedData.rename(columns={'sex':'isFemale','smoker':'isSmoker'},inplace=True)
myCleanedData.head()

In [ ]:
# 3. To check how many values of regions
myCleanedData['region'].value_counts()

In [ ]:
# Now i will do one hot encoding to reduce anamolaties
myCleanedData=pd.get_dummies(myCleanedData,columns=['region'],drop_first=True)
myCleanedData.head()

In [ ]:
# to convert this true and false in all region into 1 and 0, i will use astype function
myCleanedData=myCleanedData.astype({'region_northwest': 'int', 'region_southeast': 'int', 'region_southwest': 'int'})
myCleanedData.head()

# Feature Engineering and Extraction

**Feature Engineering**

In [ ]:
# I will create a histogram for BMI
sns.histplot(data=myCleanedData, x='bmi', bins=20)

# i will create features like overweight, underweight, normal weight, and obese based on BMI values.

In [ ]:
# Creating BMI categories
myCleanedData['bmi_category'] = pd.cut(
    myCleanedData['bmi'], 
    bins=[0, 18.5, 24.9, 29.9, float('inf')], 
    labels=['Underweight', 'Normal weight', 'Overweight', 'Obese']
    )

# float('inf') means infinity
myCleanedData

In [ ]:
# Now i will do one hot encoding to make the BMI categories binary
myCleanedData=pd.get_dummies(myCleanedData,columns=['bmi_category'],drop_first=True)
myCleanedData.head()

In [ ]:
# to convert this true and false in all bmi categories into 1 and 0, i will use astype function
myCleanedData=myCleanedData.astype(int)
myCleanedData.head()

**Feature Scaling**

In [ ]:
# for scaling, i will check the columns of myCleanedData
myCleanedData.columns

In [ ]:
# we have to do standard scaling of the data, so i will use StandardScaler from sklearn
from sklearn.preprocessing import StandardScaler

# now i will select the columns which i want to scale
myScalableColumns=['age', 'bmi', 'children']
# i will do standardization using std deviation and put it between -3 and 3

In [ ]:
# creating the  object 
myScalarObject = StandardScaler()
#and initializing the class
myCleanedData[myScalableColumns] = myScalarObject.fit_transform(myCleanedData[myScalableColumns])

myCleanedData.head()

**Feature Extraction**

In [ ]:
#Till now we haven't altered the CHARGES because it is our output variable.
#now we well see the correlation of different columns to charges.

#so for this i will use pearson correalation.
from scipy.stats import pearsonr

# ----------------------------------
# Pearson Correlation Calculation
# ----------------------------------

# List of features to check against target
mySelectedFeatures = ['age', 'bmi', 'children', 'isFemale', 
                      'isSmoker', 'region_northwest', 'region_southeast', 
                      'region_southwest', 'bmi_category_Normal weight', 
                      'bmi_category_Overweight', 'bmi_category_Obese'] 


# how my correlation looks like, so i will create a dictionary to store the correlation values
myCorrelations = {
    feature: pearsonr(myCleanedData[feature], myCleanedData['charges'])[0]
    #feature is the key, pearsonr function will give the correlation value of that feature with charges
    
    for feature in mySelectedFeatures
    # this is a for loop which will iterate through all the features in mySelectedFeatures list
} 


In [ ]:
# Now i will convert this dictionary->dataframe to ->visualize the correlation values
myCorrelationDF = pd.DataFrame(list(myCorrelations.items()), columns=['Feature', 'Correlation'])
# explanation: 
    # list(myCorrelations.items()) will give a list of tuples, 
    # each tuple will have feature and its correlation value, 
    # then we will convert this list of tuples into a dataframe with columns Feature and Correlation

# now i will sort the correlation values in descending order
myCorrelationDF.sort_values(by='Correlation', ascending=False, inplace=True)

myCorrelationDF

*FOR CATEGORICAL FEATURES: X-SQAURE TEST*

In [ ]:
#my categorical features
myCategoricalFeatures = ['isFemale', 'isSmoker', 'region_northwest', 
                         'region_southeast', 'region_southwest',
                         'bmi_category_Normal weight', 'bmi_category_Overweight', 
                         'bmi_category_Obese'
                         ]

In [ ]:
# now do the x-sqaure test

from scipy.stats import chi2_contingency
import pandas as pd

alpha = 0.05 # now this will be comapared with p-value

#now i wll create bins for charges to make it categorical, so that we can do chi-square test
myCleanedData['chargesBins'] = pd.cut(myCleanedData['charges'], 
                                       bins=5, 
                                       labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

# chi2Results is a dictionary which will store the results of chi-square test for each categorical feature with chargesBins
chi2Results = {}

#now i will iterate through all the categorical features and do the chi-square test with chargesBins
for feature in myCategoricalFeatures:
    contingencyTable = pd.crosstab(myCleanedData[feature], 
                                   myCleanedData['chargesBins'])
    chi2, p, dof, expected = chi2_contingency(contingencyTable)
    
    myDecision = 'Reject Null Hypothesis [Keep Feature]' if p < alpha else 'Accept Null Hypothesis [Remove Feature]'
    chi2Results[feature] = {
        'chi2': chi2, 
        'p-value': p, 
        'decision': myDecision}
# explanation:
    # pd.crosstab will create a contingency table for the feature and chargesBins and 
    # then we will do the chi-square test 
    # using chi2_contingency function which will give us the chi-square value, p-value, degrees of freedom and expected frequencies, 
    # then we will compare the p-value with alpha to make a decision whether to reject or accept the null hypothesis and 
    # store the results in chi2Results dictionary
    
# now we will convert this dictionary into a dataframe
chi2ResultsDF = pd.DataFrame.from_dict(chi2Results)
# taking the transpose of the dataframe to make it more readable
chi2ResultsDF = chi2ResultsDF.T
# sort according to p-value in ascending order
chi2ResultsDF.sort_values(by='p-value', ascending=True, inplace=True)
chi2ResultsDF


In [ ]:
# now removing the features which are not significant based on chi-square test and keeping only the significant features in myFinalDataFrame
myFinalDataFrame = myCleanedData.drop(columns=['region_northwest','region_southwest'])
myFinalDataFrame